In [1]:
# ============================================================
# CREDIT CARD FRAUD DETECTION
# PHASE 4 — CLASS-WEIGHTED XGBOOST
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from xgboost import XGBClassifier

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve
)


# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = Path("../data/creditcard.csv")

df = pd.read_csv(DATA_PATH)

df = df.sort_values("Time").reset_index(drop=True)

print("=" * 70)
print("DATASET")
print("=" * 70)

print(f"Shape: {df.shape}")


# ============================================================
# 2. TEMPORAL SPLIT
# ============================================================

n = len(df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:validation_end].copy()
test = df.iloc[validation_end:].copy()


# ============================================================
# 3. FEATURES / TARGET
# ============================================================

TARGET = "Class"

FEATURES = [
    column
    for column in df.columns
    if column != TARGET
]

X_train = train[FEATURES]
y_train = train[TARGET]

X_validation = validation[FEATURES]
y_validation = validation[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]


# ============================================================
# 4. CALCULATE CLASS WEIGHT
# ============================================================

negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("\n" + "=" * 70)
print("CLASS IMBALANCE")
print("=" * 70)

print(f"Legitimate training transactions : {negative:,}")
print(f"Fraud training transactions      : {positive:,}")

print(
    f"\nscale_pos_weight = "
    f"{scale_pos_weight:.2f}"
)


# ============================================================
# 5. TRAIN XGBOOST
# ============================================================

model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",

    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)


print("\nTraining XGBoost...")

model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_train, y_train),
        (X_validation, y_validation)
    ],

    verbose=False
)

print("✓ XGBoost training complete.")


# ============================================================
# 6. PREDICT PROBABILITIES
# ============================================================

train_prob = model.predict_proba(
    X_train
)[:, 1]

validation_prob = model.predict_proba(
    X_validation
)[:, 1]

test_prob = model.predict_proba(
    X_test
)[:, 1]


# ============================================================
# 7. PR-AUC
# ============================================================

train_pr_auc = average_precision_score(
    y_train,
    train_prob
)

validation_pr_auc = average_precision_score(
    y_validation,
    validation_prob
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob
)


print("\n" + "=" * 70)
print("PR-AUC")
print("=" * 70)

print(f"Train PR-AUC      : {train_pr_auc:.6f}")
print(f"Validation PR-AUC : {validation_pr_auc:.6f}")
print(f"Test PR-AUC       : {test_pr_auc:.6f}")


# ============================================================
# 8. ROC-AUC
# ============================================================

validation_roc_auc = roc_auc_score(
    y_validation,
    validation_prob
)

test_roc_auc = roc_auc_score(
    y_test,
    test_prob
)

print("\n" + "=" * 70)
print("ROC-AUC")
print("=" * 70)

print(f"Validation ROC-AUC : {validation_roc_auc:.6f}")
print(f"Test ROC-AUC       : {test_roc_auc:.6f}")


# ============================================================
# 9. THRESHOLD = 0.50
# ============================================================

threshold = 0.50

validation_pred = (
    validation_prob >= threshold
).astype(int)

test_pred = (
    test_prob >= threshold
).astype(int)


# ============================================================
# 10. TEST CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("TEST PERFORMANCE — THRESHOLD 0.50")
print("=" * 70)

print(
    classification_report(
        y_test,
        test_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 11. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    test_pred
)

tn, fp, fn, tp = cm.ravel()

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)

print(f"\nTrue Negatives  : {tn:,}")
print(f"False Positives : {fp:,}")
print(f"False Negatives : {fn:,}")
print(f"True Positives   : {tp:,}")


# ============================================================
# 12. PRECISION-RECALL CURVE
# ============================================================

precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_prob
)

plt.figure(figsize=(9, 6))

plt.plot(
    recall,
    precision,
    linewidth=2
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    f"XGBoost Precision-Recall Curve\n"
    f"Validation PR-AUC = {validation_pr_auc:.4f}"
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()


# ============================================================
# 13. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n" + "=" * 70)
print("TOP 15 FEATURES")
print("=" * 70)

display(
    importance.head(15)
)


# ============================================================
# 14. FEATURE IMPORTANCE PLOT
# ============================================================

top_features = importance.head(15).sort_values(
    "Importance"
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.xlabel("XGBoost Feature Importance")
plt.ylabel("Feature")

plt.title("Top 15 XGBoost Features")

plt.tight_layout()

plt.show()


# ============================================================
# 15. RESULTS TABLE
# ============================================================

weighted_xgb_results = pd.DataFrame({
    "Model": ["XGBoost — Class Weighted"],
    "Validation PR-AUC": [validation_pr_auc],
    "Test PR-AUC": [test_pr_auc],
    "Validation ROC-AUC": [validation_roc_auc],
    "Test ROC-AUC": [test_roc_auc],
    "Test Precision": [
        precision_score(
            y_test,
            test_pred,
            zero_division=0
        )
    ],
    "Test Recall": [
        recall_score(
            y_test,
            test_pred,
            zero_division=0
        )
    ],
    "Test F1": [
        f1_score(
            y_test,
            test_pred,
            zero_division=0
        )
    ],
    "Test Alerts": [
        int(test_pred.sum())
    ]
})

print("\n" + "=" * 70)
print("MODEL RESULTS")
print("=" * 70)

display(weighted_xgb_results)


# ============================================================
# 16. LOAD BASELINE RESULTS IF AVAILABLE
# ============================================================

baseline_path = Path(
    "../reports/baseline_results.csv"
)

if baseline_path.exists():

    baseline_results = pd.read_csv(
        baseline_path
    )

    comparison = pd.concat(
        [
            baseline_results,
            weighted_xgb_results
        ],
        ignore_index=True
    )

    print("\n" + "=" * 70)
    print("BASELINE VS XGBOOST")
    print("=" * 70)

    display(comparison)

    comparison.to_csv(
        "../reports/model_comparison.csv",
        index=False
    )

    print(
        "\n✓ Model comparison saved to "
        "../reports/model_comparison.csv"
    )


# ============================================================
# 17. SAVE MODEL
# ============================================================

model.save_model(
    "../reports/xgboost_weighted.json"
)

print(
    "\n✓ XGBoost model saved to "
    "../reports/xgboost_weighted.json"
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("PHASE 4 COMPLETE")
print("=" * 70)

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/sreeragm/fraud-decision-engine/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <09B43001-8527-34F6-AAE4-B97DA8A6CBC0> /Users/sreeragm/fraud-decision-engine/.venv/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew